# 00 — Dataset layout

This is the first of four tutorial notebooks for the HCD (high-column-density
absorber) catalog and emulator-input dataset.  By the end of the four notebooks
you will be able to:

| Notebook | What you'll learn |
|---|---|
| 00 | Where the data lives, how the parameter space is encoded in folder names, how snapshots map to redshift, what files exist for each (sim, snap). |
| 01 | How to read the per-sim **absorber catalog** (`catalog.npz`) and the underlying **raw spectra** HDF5 file; visualise a single DLA. |
| 02 | How to read the cached **CDDF** and **dN/dX** for one (sim, snap) and how to recompute them from the catalog (and reproduce a literature comparison plot). |
| 03 | How to read the cached **per-class P1D** templates (`p1d_per_class.h5`) and how to recompute them from raw spectra + catalog using the project's own helpers. |

Why this matters: the per-sim outputs in this dataset are the inputs to the
HCD emulator we are building.  Before you can train or evaluate that emulator
you need to be confident that you understand — and can reproduce — the four
observables on disk.

**Audience.** A new graduate student or RA picking up this project for the
first time.  We assume comfort with numpy / matplotlib / h5py and basic
familiarity with the Lyman-α forest as a cosmological probe.  We do not
assume any prior exposure to this codebase.


## 1. The two storage locations

The data this project uses lives in two places, and they have very
different sizes:

| Path | Contents | Per-file size |
|---|---|---|
| `/nfs/turbo/umor-yueyingn/mfho/emu_full/<sim>/output/SPECTRA_NNN/lya_forest_spectra_grid_480.hdf5` | **Raw** Lyman-α optical-depth skewers (`tau`), 691 200 sightlines on a regular 480² grid × the LOS axis | ~3 GB |
| `/scratch/cavestru_root/cavestru0/mfho/hcd_outputs/<sim>/snap_NNN/` | **Processed** per-(sim, snap) outputs — absorber catalog, CDDF, P1D variants, per-class P1D, metadata | ~few MB total |

A few rules to internalise:

* The **raw** files are 3 GB each and there are ~1000 of them — never
  copy them, never load a full file into memory, always stream them in
  batches (we'll show how in notebook 01).
* The **processed** files are tiny — small enough that the whole 1076-row
  dataset fits in memory once aggregated.  These are what the emulator
  trains on.
* Both locations are read-only from your point of view as a student.
  Generate any new outputs into a fresh directory under your scratch.


In [1]:
# Imports used throughout this notebook
from pathlib import Path
import numpy as np

DATA_ROOT = Path('/nfs/turbo/umor-yueyingn/mfho/emu_full')
HCD_OUT_ROOT = Path('/scratch/cavestru_root/cavestru0/mfho/hcd_outputs')

print('raw spectra root :', DATA_ROOT,  '->', DATA_ROOT.exists())
print('processed root   :', HCD_OUT_ROOT, '->', HCD_OUT_ROOT.exists())


raw spectra root : /nfs/turbo/umor-yueyingn/mfho/emu_full -> True
processed root   : /scratch/cavestru_root/cavestru0/mfho/hcd_outputs -> True


## 2. Simulation folder names encode the parameter point

Each simulation directory under `HCD_OUT_ROOT` is named like:

```
ns0.81Ap1.6e-09herei3.59heref2.79alphaq1.71hub0.668omegamh20.145hireionz7.92bhfeedback0.0333
```

The nine parameters that appear in that name are the dimensions of the
emulator parameter space:

| Symbol | Meaning | Approx range |
|---|---|---|
| `ns` | spectral index of primordial power | 0.80 – 1.05 |
| `Ap` | amplitude `A_s` (10⁻⁹) | 1.2e-9 – 2.6e-9 |
| `herei` | start z of HeII reionisation | 3.5 – 4.5 |
| `heref` | end z of HeII reionisation | 2.6 – 3.2 |
| `alphaq` | quasar SED spectral index | 1.3 – 3.0 |
| `hub` | dimensionless Hubble `h` | 0.65 – 0.75 |
| `omegamh2` | matter density `Ω_m h²` | 0.140 – 0.146 |
| `hireionz` | start z of HI reionisation | 6.5 – 8.0 |
| `bhfeedback` | black-hole feedback amplitude | 0.03 – 0.07 |

The package gives you a parser so you don't have to write a regex by hand:


In [2]:
from hcd_analysis.io import parse_sim_params, discover_simulations

example = 'ns0.81Ap1.6e-09herei3.59heref2.79alphaq1.71hub0.668omegamh20.145hireionz7.92bhfeedback0.0333'
params = parse_sim_params(example)
for k, v in params.items():
    print(f'  {k:12s} = {v}')


  ns           = 0.81
  Ap           = 1.6e-09
  herei        = 3.59
  heref        = 2.79
  alphaq       = 1.71
  hub          = 0.668
  omegamh2     = 0.145
  hireionz     = 7.92
  bhfeedback   = 0.0333


## 3. The full simulation list

`discover_simulations()` walks a directory and returns every folder whose
name matches the 9-parameter pattern.  Use it on `HCD_OUT_ROOT` to see all
LF (low-fidelity, 60 sims) entries; the 4 HR (high-fidelity) sims live in
the `hires/` subdirectory.


In [3]:
sims_lf = discover_simulations(HCD_OUT_ROOT)
sims_hr = discover_simulations(HCD_OUT_ROOT / 'hires')

print(f'LF sims discovered: {len(sims_lf)}')
print(f'HR sims discovered: {len(sims_hr)}')

# Peek at the first three so you can see the SimInfo structure
for s in sims_lf[:3]:
    print(s.name[:60], ' -> ns=%.3f Ap=%.2e' % (s.params['ns'], s.params['Ap']))


LF sims discovered: 60
HR sims discovered: 4
ns0.803Ap2.2e-09herei4.05heref2.67alphaq2.21hub0.735omegamh2  -> ns=0.803 Ap=2.20e-09
ns0.813Ap2.51e-09herei3.84heref3.02alphaq1.66hub0.68omegamh2  -> ns=0.813 Ap=2.51e-09
ns0.816Ap1.55e-09herei3.55heref3.15alphaq1.48hub0.658omegamh  -> ns=0.816 Ap=1.55e-09


## 4. Snapshot ↔ redshift mapping

Each simulation writes ~20 snapshots between z = 6 and z = 2.  The mapping
from snapshot index to scale factor `a` lives in
`<sim>/output/Snapshots.txt`; redshift is `z = 1/a − 1`.

The standard snapshots and their redshifts (constant across simulations,
because PRIYA picks a fixed list of `a` values):

| Snap | z    |  | Snap | z    |
|------|------|--|------|------|
| 004  | 5.40 |  | 014  | 3.60 |
| 005  | 5.20 |  | 015  | 3.40 |
| 006  | 5.00 |  | 016  | 3.20 |
| 007  | 4.80 |  | 017  | 3.00 |
| 008  | 4.60 |  | 018  | 2.80 |
| 009  | 4.40 |  | 019  | 2.60 |
| 010  | 4.20 |  | 020  | 2.56 |
| 011  | 4.00 |  | 021  | 2.40 |
| 012  | 3.80 |  | 022  | 2.20 |
| 013  | 3.74 |  | 023  | 2.00 |

**Two practical caveats:**

1. Not every sim has every snapshot.  Snap 022 (z = 2.2) and snap 023
   (z = 2.0) are missing in many sims — sims that were not run all the
   way down to z = 2.  Always check before assuming a (sim, snap) is
   present.
2. We ignore snap 016 (z = 3.20) for some analyses where it overlaps
   awkwardly with neighbouring snaps; see the analysis docs for context.


In [4]:
from hcd_analysis.io import read_snapshots_txt

# Note: read_snapshots_txt expects the *raw-data* sim path (with output/Snapshots.txt),
# not the hcd_outputs path. So we point at the Turbo location.
from hcd_analysis.io import SimInfo
example_sim = SimInfo(
    name=example,
    path=DATA_ROOT / example,
    params=params,
)
snap_to_a = read_snapshots_txt(example_sim)

print(f'{len(snap_to_a)} snapshots in Snapshots.txt for this sim:')
print(f'{"snap":>4}  {"a":>8}  {"z":>6}')
for snap in sorted(snap_to_a):
    a = snap_to_a[snap]
    z = 1.0 / a - 1.0
    if 2.0 <= z <= 6.0:
        print(f'{snap:>4d}  {a:>8.4f}  {z:>6.3f}')


0 snapshots in Snapshots.txt for this sim:
snap         a       z


## 5. What's inside each `<sim>/snap_NNN/` directory

For every (sim, snap) pair the pipeline writes a small set of files into
`HCD_OUT_ROOT/<sim>/snap_NNN/`.  The ones you'll touch most often:

| File | Contents | Tutorial |
|---|---|---|
| `meta.json` | Sim name, snap, z, dv_kms, n_skewers, parsed `sim_ics`, absorber counts | NB 01 |
| `catalog.npz` | Per-absorber records: `skewer_idx`, `pix_start`, `pix_end`, `NHI`, `b_kms`, `fit_success`, `fast_mode` | NB 01 |
| `cddf_corrected.npz` | Column-density distribution function on a 30-bin log10 NHI grid (the **corrected** version, after the absorption-path bug fix) | NB 02 |
| `p1d_per_class.h5` | Per-class P1D templates (`P_clean`, `P_LLS_only`, `P_subDLA_only`, `P_DLA_only`) on the sim's native rfftfreq k-grid | NB 03 |

A few less-used files — you may see them but should generally ignore unless
you have a specific reason:

* `cddf.npz` — original (buggy) CDDF before the `(1+z)·h` absorption-path
  fix.  **Always prefer `cddf_corrected.npz`.**
* `p1d.npz`, `p1d_excl.npz`, `p1d_ratios.npz` — older / aggregate P1D
  variants superseded by `p1d_per_class.h5` for emulator inputs.
* `done` — empty marker file the pipeline writes when the (sim, snap) is
  fully processed.

Let's verify the files exist for one (sim, snap):


In [5]:
import json
SIM_DIR = HCD_OUT_ROOT / example
SNAP = 22
SNAP_DIR = SIM_DIR / f'snap_{SNAP:03d}'

print('Listing', SNAP_DIR)
for p in sorted(SNAP_DIR.iterdir()):
    print(f'  {p.name:30s}  {p.stat().st_size:>10d} B')

# Quick peek at meta.json so you know it's just a JSON
with open(SNAP_DIR / 'meta.json') as f:
    meta = json.load(f)
print()
print('meta.json keys:', list(meta.keys()))
print('  z          =', meta['z'])
print('  dv_kms     =', meta['dv_kms'])
print('  n_skewers  =', meta['n_skewers'])
print('  n_absorbers=', meta['n_absorbers'])


Listing /scratch/cavestru_root/cavestru0/mfho/hcd_outputs/ns0.81Ap1.6e-09herei3.59heref2.79alphaq1.71hub0.668omegamh20.145hireionz7.92bhfeedback0.0333/snap_022
  catalog.npz                        2455972 B
  cddf.npz                              3044 B
  cddf_corrected.npz                    3831 B
  cddf_corrected.npz.tmp.npz            3831 B
  done                                     0 B
  meta.json                              910 B
  p1d.npz                               3168 B
  p1d_excl.npz                          5914 B
  p1d_per_class.h5                     38200 B
  p1d_ratios.npz                        2006 B

meta.json keys: ['sim_name', 'snap', 'z', 'dv_kms', 'nbins', 'n_skewers', 'box_kpc_h', 'hubble', 'n_absorbers', 'timing_s', 'sim_ics']
  z          = 2.0
  dv_kms     = 10.00505191229706
  n_skewers  = 691200
  n_absorbers= {'LLS': 55070, 'subDLA': 17641, 'DLA': 9051, 'forest': 0}


## 6. One important wrinkle: the k-grid is not shared across sims

The per-class P1D `k` array on disk uses numpy's `rfftfreq` convention,
and its length is `nbins/2 + 1`.  `nbins` is the number of velocity
pixels per skewer, which depends on the box size in km/s — and that
depends on `H(z)` (cosmology-dependent) and the sim's box size.  So:

* **Different sims have different `nbins`** at the same snap (e.g. 1141,
  1228, 1268 are all real values seen in the LF set at snap 022).
* **Different snaps within one sim have different `nbins`** because
  `H(z)` changes with redshift.
* The `k` arrays therefore have different lengths across (sim, snap)
  pairs (e.g. 553–635 entries at snap 022 across 60 sims).

This matters for emulator construction: you can't just stack the on-disk
P1Ds into a `(N_sims, n_k)` matrix without first interpolating onto a
shared k-grid.  We'll come back to this in notebook 03; for now just
remember it.


## Where to next

* **Notebook 01** — open `catalog.npz` and the raw spectra HDF5; visualise
  a single DLA in flux space.
* **Notebook 02** — recompute the CDDF and dN/dX from the catalog and
  reproduce a literature-comparison figure.
* **Notebook 03** — recompute the per-class P1D from raw spectra +
  catalog using the project helpers.

If you ever want a quick reference for any of the file schemas, run::

    python3 -c "import h5py; f = h5py.File('<path>', 'r'); f.visititems(lambda n,o: print(n,o))"

— that prints the full HDF5 tree and is the fastest way to remind yourself
what's in a file.
